# **Grocery Store Dataset Cleaning and Analysis**
###This notebook will walk through the loading, cleaning, and exporting of the grocery_chain_data dataset available on Kaggle.
 https://www.kaggle.com/datasets/pratyushpuri/grocery-store-sales-dataset-in-2025-1900-record
###In this notebook we will load the .csv file into a pandas dataframe, do some initial exploration and analysis, perform some data cleaning and feature engineering, and finally export the cleaned dataset to create some data visualizations in Tableau.

In [ ]:
#Download the grocery_chain_data dataset from Kaggle to run this notebook
#It may download as a zipped folder, you will need to be able to access the grocery_chain_data.csv file within that folder

# https://www.kaggle.com/datasets/pratyushpuri/grocery-store-sales-dataset-in-2025-1900-record

#You can hit the "Run all" button at the top of this notebook to have all the cell outputs generate at once.

import pandas as pd
from google.colab import files
uploaded = files.upload()

raw_df = pd.read_csv("grocery_chain_data.csv")

In [ ]:
print(raw_df.head())

###This cell displays the first 5 rows of the dataset so we can see column names, number of columns, and types of data we will be working with.

# Data Cleaning and Feature Engineering


In [ ]:
raw_df.info()

###This cell shows the datatypes stored in each column and the number of non-null values each column has.

###The store_name column does not have the same number of non-null values as the other columns, indicating some empty cells (1,955 vs. 1,980). We will look into this further.


###

In [ ]:
raw_df.isnull().sum()

### We will input the value "unknown" for records with a missing value for the store_name column. This will make it easy to understand for downstream audiences of this dataset that these records are missing their store_name value.


### Inputting a value for these records will also allow us to keep as much data as possible for analysis on other features.


### This input will also make this population easy to query when performing analysis or trends on why these records are possibly missing their store_name value.

In [ ]:
raw_df_v1 = raw_df.copy()
raw_df_v1["store_name"] = raw_df_v1["store_name"].fillna("Unknown")

In [ ]:
raw_df_v1.info()

### This dataset also features negative values for the "final_amount" column. This could be caused by errors in the original pricing of the item or the discounting of items.


In [ ]:
print(raw_df_v1.describe())

###In the cell above, the minimum value for the final_amount column is negative.

In [ ]:
(raw_df_v1['final_amount'] <= 0).sum()

In [ ]:
raw_df_v1[raw_df_v1['final_amount'] <= 0]

### There are 13 such records with a negative final_amount value. Looking over the records with negative final amounts, it doesn't appear to be strongly affecting any particular store, aisle, or product. There are also no duplicate customer id's in this subset, suggesting it is a random error.

### Similar to the previous step, I would like to keep as much data as possible. I don't want to input 0 as a value for these records, since I would like to keep the original final_amount value for later root cause analysis into this issue. I will make a new column that reports the original final_amount column if it is above 0, and 0 if it has a negative value. This new column will give us better data and summary statistics regarding what customers actually paid for items.

###We will call this new column **final_amount_clean**

In [ ]:
raw_df_v1["final_amount_clean"] = raw_df_v1["final_amount"].clip(lower=0)

In [ ]:
print(raw_df_v1.head())

###The minimum of the new final_amount_clean column is 0 meaning we have addressed that issue. Lets take a look at the summary statistics for that column.


In [ ]:
print(raw_df_v1.describe())

### That new column does not have largely different summary statistics, but understanding that column and using it to create visualizations will make a lot more sense now with the minimum value being 0.

###We are going to perform some final data cleaning steps in the cells below.




In [ ]:
raw_df_v1['transaction_date'] = pd.to_datetime(raw_df_v1['transaction_date'])

###The cell above ensures all values in the transaction_date column are in the proper datetime format. This will make sure any visualizations we create that use time as a variable will be properly formatted.

In [ ]:
raw_df_v1['month'] = raw_df_v1['transaction_date'].dt.to_period('M').astype(str)

### The cell above creates another column that tracks the month of each transaction date. This could be helpful for measuring monthly trends within the data.

In [ ]:
raw_df_v1.sample(n=5, random_state=1)

### The cell above takes a random sample of 5 entries from the data so we can see what the final column layout and datatypes look like. Using the same **random_state** value ensures we will always get this same sample, so this parameter can be changed if we want a truly random sample.

# Exploration/Aggregation
###We'll do some SQL like exploration into the cleaned dataset to generate some ideas for visualizations. First lets look at the records with the largest transaction amounts.


In [ ]:
raw_df_v1.nlargest(5,"final_amount_clean")

###I think it would be interesting to do a deepdive on the leading customers in terms of transaction amount (final_amount_clean), loyalty points, and quantity. We'll pull the top 5 for each of these categories.



In [ ]:
top_customers_sales = (
    raw_df_v1.groupby("customer_id", as_index=False)["final_amount_clean"]
    .sum()
    .nlargest(5, "final_amount_clean"))
top_customers_sales

In [ ]:
top_customers_loyalty = (
    raw_df_v1.groupby("customer_id", as_index=False)["loyalty_points"]
    .sum()
    .nlargest(5, "loyalty_points"))
top_customers_loyalty

In [ ]:
top_customers_quantity = (
    raw_df_v1.groupby("customer_id", as_index=False)["quantity"]
    .sum()
    .nlargest(5, "quantity"))
top_customers_quantity

###We can look to see if any of the same customer IDs pop up in these lists, or we can use some code to make that part easy for us. This cell makes a new dataframe that records the customer id and which of the top 5 lists it appears in. Then we'll look at a summary that details how many lists each customer appears in, which specific lists those are, and sorts those results to show the most “frequent” customers.

In [ ]:
combined = pd.concat([
    top_customers_sales.assign(source="sales")[["customer_id", "source"]],
    top_customers_loyalty.assign(source="loyalty")[["customer_id", "source"]],
    top_customers_quantity.assign(source="quantity")[["customer_id", "source"]],
])

overlap_detail = (
    combined.groupby("customer_id")["source"]
    .agg([
        ("count", "count"),
        ("sources", lambda x: ", ".join(sorted(x)))
    ])
    .reset_index()
    .sort_values(by="count", ascending=False)
)

overlap_detail.head()

###Looks like customers 2276, 7658, and 8381 are some of the "rock star" customers from this dataset in terms of total loyalty points, sales, and quantities.


# Export


In [ ]:
raw_df_v1.rename(columns={"final_amount_clean": "sales"}, inplace=True)

###We'll rename our final_amount_clean column to sales before exporting. This will make that column easier to interpret in any visualizations we create.

In [ ]:
cleaned_df = raw_df_v1.copy()
cleaned_df.to_csv("grocery_store_data_cleaned.csv", index=False)

In [ ]:
cleaned_df.head()

### We'll make a copy of this cleaned and transformed dataset to use in visualizations. The cell above will create the copy. The cell below will allow us to export and download this copy to use in Tableau.

In [ ]:
from google.colab import files
files.download("grocery_store_data_cleaned.csv")